# Word-Level Sign Language Detection (Temporal CNN + MediaPipe)

This notebook builds a full pipeline that converts labeled ASL word videos into MediaPipe hand landmarks, stores them as fixed-length sequences, trains a temporal CNN classifier, and provides helpers for running inference on new clips. Every line is annotated so you can follow the logic end-to-end.


In [78]:
# !pip install mediapipe opencv-python torch torchvision torchaudio pandas scikit-learn  # Uncomment if dependencies are missing

In [1]:
import json  # Handles reading and writing label maps
import math  # Provides helpers for ceiling/floor operations during sampling
import os  # Offers operating-system path utilities if needed
from dataclasses import dataclass  # Makes configuration containers concise
from pathlib import Path  # Enables clean path manipulations
from typing import Dict, List, Tuple  # Gives precise type annotations for clarity
from collections import deque  # Maintains a rolling buffer for live sequences

import cv2  # Loads and decodes video frames
import mediapipe as mp  # Supplies pretrained landmark detectors
import numpy as np  # Enables fast numerical tensor ops
import torch  # Hosts tensors and neural nets
from sklearn.model_selection import train_test_split  # Splits datasets deterministically
from torch import nn  # Provides neural network layers
from torch.utils.data import DataLoader, Dataset  # Supplies PyTorch dataset abstractions

In [2]:
@dataclass
class WordLevelConfig:
    """Holds all file paths, preprocessing knobs, and training hyperparameters."""

    videos_root: Path = Path("/home/kaleab/Desktop/tiny_WLASL_dataset_processed")  # Root folder with word-named subfolders
    processed_dir: Path = Path("data")  # Destination for cached sequences
    sequence_file: str = "sequences.npy"  # File name for the numpy array storing sequences
    label_file: str = "labels.npy"  # File name for the numpy array storing integer labels
    label_map_json: str = "label_map.json"  # JSON file capturing word-to-index mapping
    video_extensions: Tuple[str, ...] = (".mp4", ".mov", ".mkv", ".avi", ".webm")  # Allowed video extensions
    sequence_length: int = 48  # Number of frames sampled per clip
    num_hands: int = 2  # MediaPipe Hands tracks left and right hands separately
    num_keypoints: int = 21  # Each hand yields 21 landmarks
    landmark_dims: int = 3  # Each landmark stores x, y, z
    min_detection_confidence: float = 0.7  # Threshold for MediaPipe detections
    min_tracking_confidence: float = 0.7  # Tracking confidence to keep detections stable
    batch_size: int = 8  # Number of sequences per training batch
    num_epochs: int = 60  # Training epochs
    learning_rate: float = 1e-3  # Adam learning rate
    dropout: float = 0.3  # Dropout for the temporal CNN
    hidden_channels: Tuple[int, ...] = (128, 256)  # Temporal CNN channel sizes
    kernel_size: int = 3  # Temporal convolution kernel width
    test_size: float = 0.2  # Validation split fraction
    random_state: int = 42  # RNG seed for reproducibility
    device: str = "cuda" if torch.cuda.is_available() else "cpu"  # Prefer GPU when available

In [3]:
config = WordLevelConfig()  # Instantiate configuration with defaults
config.processed_dir.mkdir(parents=True, exist_ok=True)  # Ensure cache directory exists before use
print(f"Using device: {config.device}")  # Report which accelerator is active for training

mp_hands = mp.solutions.hands  # Shortcut to MediaPipe Hands for reuse

Using device: cpu


In [4]:
def normalize_landmarks(landmarks: np.ndarray) -> np.ndarray:
    """Center and scale landmarks so the model ignores absolute position."""

    centered = landmarks - landmarks.mean(axis=0, keepdims=True)  # Move centroid to origin
    norm = np.linalg.norm(centered) + 1e-6  # Compute magnitude with epsilon guarding zero
    normalized = centered / norm  # Scale coordinates to unit length
    return normalized  # Return normalized landmarks


def extract_hand_features(frame_rgb: np.ndarray, hands: mp.solutions.hands.Hands) -> np.ndarray:
    """Run MediaPipe on one RGB frame and return flattened left/right hand features."""

    results = hands.process(frame_rgb)  # Execute landmark detection on the current frame
    hand_array = np.zeros((config.num_hands, config.num_keypoints, config.landmark_dims), dtype=np.float32)  # Allocate output tensor
    if results.multi_hand_landmarks and results.multi_handedness:  # Proceed only if hands detected
        for landmark_set, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):  # Iterate detections with labels
            coords = np.array([[lm.x, lm.y, lm.z] for lm in landmark_set.landmark], dtype=np.float32)  # Collect coordinates
            coords = normalize_landmarks(coords)  # Normalize coordinates for invariance
            label = handedness.classification[0].label.lower()  # Determine if detection is left or right hand
            idx = 0 if label == "left" else 1  # Map label to array index (0=left,1=right)
            hand_array[idx] = coords  # Store normalized coordinates in the slot for that hand
    flat = hand_array.reshape(-1)  # Flatten both hands into a single feature vector
    return flat  # Return flattened representation for this frame

In [5]:
def sample_video_frames(video_path: Path, num_frames: int) -> List[np.ndarray]:
    """Read a video and return a uniformly sampled list of RGB frames."""

    cap = cv2.VideoCapture(str(video_path))  # Open the video file with OpenCV
    if not cap.isOpened():  # Validate successful open
        raise RuntimeError(f"Could not open video: {video_path}")  # Raise descriptive error
    frames: List[np.ndarray] = []  # Collect decoded frames in memory
    while True:  # Loop until frames end
        ret, frame = cap.read()  # Read the next frame
        if not ret:  # Stop if decoding failed or video ended
            break  # Exit loop when no frames remain
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB for MediaPipe
        frames.append(rgb)  # Append the converted frame to the list
    cap.release()  # Release the capture resource immediately
    if not frames:  # Guard against empty videos
        raise RuntimeError(f"No frames decoded from {video_path}")  # Surface issue to the caller
    indices = np.linspace(0, len(frames) - 1, num_frames).astype(int)  # Select evenly spaced indices
    sampled_frames = [frames[i] for i in indices]  # Gather sampled frames
    return sampled_frames  # Return the subset sized to sequence_length


def video_to_landmark_sequence(video_path: Path, hands: mp.solutions.hands.Hands) -> np.ndarray:
    """Convert an entire video into a fixed-length (T x F) landmark sequence."""

    sampled_frames = sample_video_frames(video_path, config.sequence_length)  # Grab evenly spaced frames
    features = [extract_hand_features(frame, hands) for frame in sampled_frames]  # Convert each frame to landmarks
    sequence = np.stack(features, axis=0).astype(np.float32)  # Stack features into a tensor
    return sequence  # Return sequence ready for caching

In [6]:
def generate_sequence_cache(overwrite: bool = False) -> None:
    """Run MediaPipe over the dataset and store landmark sequences + labels."""

    seq_path = config.processed_dir / config.sequence_file  # Path for the sequence numpy file
    label_path = config.processed_dir / config.label_file  # Path for the label numpy file
    map_path = config.processed_dir / config.label_map_json  # Path for the label map JSON
    if seq_path.exists() and label_path.exists() and map_path.exists() and not overwrite:  # Respect existing cache
        print("Cached sequences already exist. Set overwrite=True to rebuild.")  # Inform user about reuse
        return  # Early exit to save time

    label_map: Dict[str, int] = {}  # Map each word to an integer index
    sequences: List[np.ndarray] = []  # Store landmark sequences
    labels: List[int] = []  # Store corresponding integer labels

    with mp_hands.Hands(  # Initialize MediaPipe once for efficiency
        static_image_mode=False,
        max_num_hands=config.num_hands,
        min_detection_confidence=config.min_detection_confidence,
        min_tracking_confidence=config.min_tracking_confidence,
    ) as hands:
        for class_dir in sorted(config.videos_root.iterdir()):  # Iterate word folders alphabetically
            if not class_dir.is_dir():  # Skip non-folder entries
                continue  # Ignore stray files
            label_name = class_dir.name.strip().upper()  # Normalize folder name to uppercase
            if label_name not in label_map:  # Assign new labels sequentially
                label_map[label_name] = len(label_map)  # Assign next integer id
            label_index = label_map[label_name]  # Fetch the assigned integer id
            video_files = [p for p in sorted(class_dir.iterdir()) if p.suffix.lower() in config.video_extensions]  # Filter valid videos
            if not video_files:  # Warn if folder empty
                print(f"No supported videos found in {class_dir}")  # Log empty folder
                continue  # Move to the next class
            for video_path in video_files:  # Process each clip for this word
                try:
                    sequence = video_to_landmark_sequence(video_path, hands)  # Convert clip to landmarks
                except RuntimeError as err:  # Catch decoding issues
                    print(f"Skipping {video_path}: {err}")  # Report failure and continue
                    continue  # Skip to next video
                sequences.append(sequence)  # Cache the sequence
                labels.append(label_index)  # Cache the label
                print(f"Processed {video_path} -> sequence shape {sequence.shape}")  # Provide progress feedback

    if not sequences:  # Ensure we captured at least one sequence
        raise RuntimeError("No sequences were generated. Check dataset paths and formats.")  # Fail loudly when empty

    sequence_array = np.stack(sequences).astype(np.float32)  # Convert list to contiguous numpy array
    label_array = np.array(labels, dtype=np.int64)  # Convert list to numpy vector
    np.save(seq_path, sequence_array)  # Persist sequences to disk
    np.save(label_path, label_array)  # Persist labels to disk
    map_path.write_text(json.dumps(label_map, indent=2))  # Save label map for inference use
    print(f"Saved sequences to {seq_path} with shape {sequence_array.shape}")  # Confirm save path
    print(f"Saved labels to {label_path} with shape {label_array.shape}")  # Confirm labels path
    print(f"Label map saved to {map_path} with {len(label_map)} entries")  # Report number of classes

In [7]:
def load_cached_sequences() -> Tuple[np.ndarray, np.ndarray, Dict[str, int]]:
    """Load cached numpy arrays along with the label map."""

    seq_path = config.processed_dir / config.sequence_file  # Locate sequences file
    label_path = config.processed_dir / config.label_file  # Locate labels file
    map_path = config.processed_dir / config.label_map_json  # Locate label map JSON
    if not seq_path.exists() or not label_path.exists() or not map_path.exists():  # Verify cache presence
        raise FileNotFoundError("Cache files missing. Run generate_sequence_cache() first.")  # Prompt preprocessing
    sequences = np.load(seq_path)  # Load sequence array into memory
    labels = np.load(label_path)  # Load label vector into memory
    label_map = json.loads(map_path.read_text())  # Load label map dictionary
    return sequences, labels, label_map  # Return all three artifacts


class WordSignDataset(Dataset):
    """Simple Dataset wrapper around precomputed landmark sequences."""

    def __init__(self, sequences: np.ndarray, labels: np.ndarray):
        self.sequences = torch.from_numpy(sequences)  # Convert numpy array to tensor once
        self.labels = torch.from_numpy(labels)  # Convert labels to tensor once

    def __len__(self) -> int:
        return len(self.labels)  # Return dataset size

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.sequences[idx], self.labels[idx]  # Provide tensors ready for training


def create_dataloaders() -> Tuple[DataLoader, DataLoader, Dict[str, int]]:
    """Split cached data into train/val sets and wrap them in DataLoaders."""

    sequences, labels, label_map = load_cached_sequences()  # Load cached arrays
    train_seq, val_seq, train_labels, val_labels = train_test_split(  # Stratified split
        sequences,
        labels,
        test_size=config.test_size,
        random_state=config.random_state,
        stratify=labels,
    )
    train_dataset = WordSignDataset(train_seq, train_labels)  # Wrap training data
    val_dataset = WordSignDataset(val_seq, val_labels)  # Wrap validation data
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)  # Shuffle training batches
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)  # Keep validation deterministic
    return train_loader, val_loader, label_map  # Provide loaders plus label map

In [8]:
class TemporalConvNet(nn.Module):
    """Lightweight temporal CNN classifier operating on landmark sequences."""

    def __init__(self, feature_dim: int, num_classes: int):
        super().__init__()  # Initialize nn.Module internals
        layers: List[nn.Module] = []  # Collect sequential temporal layers
        in_channels = feature_dim  # Initial channel count equals feature dimension
        for out_channels in config.hidden_channels:  # Iterate desired channel sizes
            layers.append(nn.Conv1d(in_channels, out_channels, kernel_size=config.kernel_size, padding=config.kernel_size // 2))  # Temporal convolution
            layers.append(nn.BatchNorm1d(out_channels))  # Stabilize activations
            layers.append(nn.ReLU())  # Apply non-linearity
            layers.append(nn.Dropout(config.dropout))  # Regularize activations
            in_channels = out_channels  # Update for next block
        self.encoder = nn.Sequential(*layers)  # Bundle temporal stack
        self.classifier = nn.Linear(in_channels, num_classes)  # Final linear head

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)  # Rearrange to (batch, features, time)
        encoded = self.encoder(x)  # Apply temporal convolutions
        pooled = encoded.mean(dim=-1)  # Average over time to get clip embedding
        logits = self.classifier(pooled)  # Map embedding to class logits
        return logits  # Return logits for loss computation

In [9]:
def train_one_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: torch.optim.Optimizer) -> Tuple[float, float]:
    """Run one training epoch and return average loss and accuracy."""

    model.train()  # Enable training mode for dropout/batchnorm
    total_loss = 0.0  # Accumulate loss
    total_correct = 0  # Accumulate correct predictions
    total_samples = 0  # Accumulate sample count
    for sequences, labels in loader:  # Iterate batches
        sequences = sequences.to(config.device)  # Move sequences to device
        labels = labels.to(config.device)  # Move labels to device
        optimizer.zero_grad()  # Clear stale gradients
        logits = model(sequences)  # Forward pass through network
        loss = criterion(logits, labels)  # Compute cross-entropy loss
        loss.backward()  # Backpropagate gradients
        optimizer.step()  # Update weights
        preds = logits.argmax(dim=1)  # Get predicted class indices
        total_loss += loss.item() * sequences.size(0)  # Accumulate scaled loss
        total_correct += (preds == labels).sum().item()  # Count correct predictions
        total_samples += sequences.size(0)  # Track sample count
    avg_loss = total_loss / total_samples  # Compute mean loss
    avg_acc = total_correct / total_samples  # Compute accuracy
    return avg_loss, avg_acc  # Return metrics for logging


def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> Tuple[float, float]:
    """Evaluate the model on validation data."""

    model.eval()  # Switch to eval mode
    total_loss = 0.0  # Accumulate loss
    total_correct = 0  # Accumulate accuracy counts
    total_samples = 0  # Accumulate sample count
    with torch.no_grad():  # Disable gradients for evaluation
        for sequences, labels in loader:  # Iterate validation batches
            sequences = sequences.to(config.device)  # Move sequences to device
            labels = labels.to(config.device)  # Move labels to device
            logits = model(sequences)  # Forward pass
            loss = criterion(logits, labels)  # Compute loss
            preds = logits.argmax(dim=1)  # Predicted labels
            total_loss += loss.item() * sequences.size(0)  # Accumulate scaled loss
            total_correct += (preds == labels).sum().item()  # Count correct
            total_samples += sequences.size(0)  # Track number of samples
    avg_loss = total_loss / total_samples  # Mean loss
    avg_acc = total_correct / total_samples  # Mean accuracy
    return avg_loss, avg_acc  # Return metrics


def train_model(train_loader: DataLoader, val_loader: DataLoader, num_classes: int) -> nn.Module:
    """Full training loop with best-checkpoint tracking."""

    feature_dim = config.num_hands * config.num_keypoints * config.landmark_dims  # Compute per-frame feature size
    model = TemporalConvNet(feature_dim, num_classes).to(config.device)  # Instantiate model on device
    criterion = nn.CrossEntropyLoss()  # Standard classification loss
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)  # Adam optimizer
    best_val_acc = 0.0  # Track best validation accuracy
    best_state = None  # Store state_dict of best model
    for epoch in range(1, config.num_epochs + 1):  # Iterate epochs
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)  # Train epoch
        val_loss, val_acc = evaluate(model, val_loader, criterion)  # Validate epoch
        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss {train_loss:.4f} | Train Acc {train_acc:.3f} | "
            f"Val Loss {val_loss:.4f} | Val Acc {val_acc:.3f}"
        )  # Log metrics for monitoring
        if val_acc > best_val_acc:  # Check for improvement
            best_val_acc = val_acc  # Update best accuracy
            best_state = model.state_dict()  # Snapshot weights
    if best_state is not None:  # Restore best checkpoint after training
        model.load_state_dict(best_state)  # Load best weights into model
    return model  # Return trained model ready for inference

In [10]:
def save_model(model: nn.Module, label_map: Dict[str, int]) -> Path:
    """Persist trained weights and label map side-by-side."""

    weights_path = config.processed_dir / "word_level_tcn.pt"  # Choose checkpoint filename
    torch.save(model.state_dict(), weights_path)  # Save weights to disk
    map_path = config.processed_dir / config.label_map_json  # Reuse label map path
    print(f"Saved model weights to {weights_path}")  # Notify user
    print(f"Label map remains at {map_path}")  # Remind where labels live
    return weights_path  # Return the checkpoint path for reference


def load_model(label_map: Dict[str, int]) -> nn.Module:
    """Reload a saved model for inference."""

    weights_path = config.processed_dir / "word_level_tcn.pt"  # Determine checkpoint path
    if not weights_path.exists():  # Ensure checkpoint exists
        raise FileNotFoundError("Trained weights not found. Train the model before loading.")  # Prompt training
    feature_dim = config.num_hands * config.num_keypoints * config.landmark_dims  # Feature dimension per frame
    num_classes = len(label_map)  # Determine head size from label map
    model = TemporalConvNet(feature_dim, num_classes).to(config.device)  # Instantiate model
    state_dict = torch.load(weights_path, map_location=config.device)  # Load state dict
    model.load_state_dict(state_dict)  # Restore weights
    model.eval()  # Set to evaluation mode
    return model  # Return ready-to-use model


def predict_video(video_path: Path, model: nn.Module, label_map: Dict[str, int]) -> Tuple[str, float]:
    """Run inference on a single video clip and return predicted word + confidence."""

    reversed_map = {idx: label for label, idx in label_map.items()}  # Reverse mapping for readability
    with mp_hands.Hands(  # Initialize MediaPipe for this ad-hoc prediction
        static_image_mode=False,
        max_num_hands=config.num_hands,
        min_detection_confidence=config.min_detection_confidence,
        min_tracking_confidence=config.min_tracking_confidence,
    ) as hands:
        sequence = video_to_landmark_sequence(video_path, hands)  # Convert clip to landmarks
    tensor = torch.from_numpy(sequence).unsqueeze(0).to(config.device)  # Add batch dimension and move to device
    with torch.no_grad():  # Disable gradients for inference
        logits = model(tensor)  # Forward pass
        probs = torch.softmax(logits, dim=1)  # Convert logits to probabilities
        conf, pred_idx = torch.max(probs, dim=1)  # Extract confidence and index
    predicted_word = reversed_map.get(int(pred_idx.item()), "UNKNOWN")  # Translate index to label
    confidence = float(conf.item())  # Convert confidence to Python float
    return predicted_word, confidence  # Return tuple for downstream use

In [11]:
def full_training_run(overwrite_cache: bool = False) -> nn.Module:
    """High-level helper to preprocess, train, and save the model."""

    generate_sequence_cache(overwrite=overwrite_cache)  # Ensure cached sequences exist
    train_loader, val_loader, label_map = create_dataloaders()  # Build loaders and retrieve labels
    model = train_model(train_loader, val_loader, num_classes=len(label_map))  # Train model
    save_model(model, label_map)  # Persist checkpoint to disk
    return model  # Return trained instance for immediate inference


def demo_prediction_clip(video_path: str) -> None:
    """Utility to load the saved model and print a prediction for a given clip."""

    _, _, label_map = load_cached_sequences()  # Reload label map without re-reading sequences entirely
    model = load_model(label_map)  # Load trained weights
    predicted_word, confidence = predict_video(Path(video_path), model, label_map)  # Run inference
    print(f"Predicted word: {predicted_word} (confidence {confidence:.2f})")  # Display results

In [12]:
def run_live_webcam_inference(camera_index: int = 0) -> None:
    """Stream webcam frames, maintain a rolling sequence buffer, and display predictions."""

    _, _, label_map = load_cached_sequences()  # Reload label map so predictions map to words
    model = load_model(label_map)  # Load the trained temporal CNN weights
    reversed_map = {idx: label for label, idx in label_map.items()}  # Reverse mapping for readable labels
    buffer = deque(maxlen=config.sequence_length)  # Rolling buffer that stores the latest frame features

    cap = cv2.VideoCapture(camera_index)  # Open the webcam device
    if not cap.isOpened():  # Verify the webcam can be accessed
        raise RuntimeError("Could not access webcam")  # Fail fast when camera is unavailable

    with mp_hands.Hands(  # Initialize MediaPipe Hands once for the entire session
        static_image_mode=False,
        max_num_hands=config.num_hands,
        min_detection_confidence=config.min_detection_confidence,
        min_tracking_confidence=config.min_tracking_confidence,
    ) as hands:
        while True:  # Process frames until the user quits
            ret, frame = cap.read()  # Capture the next frame from the webcam
            if not ret:  # Exit if frame capture fails
                print("Frame grab failed, exiting stream.")  # Inform the user about the issue
                break  # Break out of the loop when frames stop

            frame = cv2.flip(frame, 1)  # Mirror the frame for a natural user experience
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # Convert to RGB for MediaPipe processing
            features = extract_hand_features(rgb_frame, hands)  # Convert the current frame to landmark features
            buffer.append(features)  # Push features into the rolling buffer

            if len(buffer) == config.sequence_length:  # Only predict when the buffer is full
                sequence = np.stack(buffer, axis=0).astype(np.float32)  # Stack buffered frames into a tensor
                tensor = torch.from_numpy(sequence).unsqueeze(0).to(config.device)  # Add batch dimension and move to device
                with torch.no_grad():  # Disable gradients for inference
                    logits = model(tensor)  # Forward pass through the model
                    probs = torch.softmax(logits, dim=1)  # Convert logits to probabilities
                    conf, pred_idx = torch.max(probs, dim=1)  # Extract confidence and predicted index
                predicted_word = reversed_map.get(int(pred_idx.item()), "UNKNOWN")  # Map index back to a word
                confidence = float(conf.item())  # Convert confidence to Python float
                overlay_text = f"{predicted_word} ({confidence:.2f})"  # Format overlay string
            else:  # When buffer not full yet
                overlay_text = "Collecting frames..."  # Prompt the user to keep signing

            cv2.putText(  # Draw the overlay text on the frame
                frame,
                overlay_text,
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.1,
                (0, 255, 0),
                2,
                cv2.LINE_AA,
            )
            cv2.imshow("Word-Level ASL Live Inference", frame)  # Display the annotated frame
            if cv2.waitKey(1) & 0xFF == ord('q'):  # Allow the user to quit by pressing 'q'
                break  # Exit the loop when requested

    cap.release()  # Release the webcam resource
    cv2.destroyAllWindows()  # Close any OpenCV windows that were opened


In [13]:
model = full_training_run()

Cached sequences already exist. Set overwrite=True to rebuild.
Epoch 01 | Train Loss 2.5876 | Train Acc 0.282 | Val Loss 2.7834 | Val Acc 0.395
Epoch 02 | Train Loss 1.8197 | Train Acc 0.604 | Val Loss 2.1978 | Val Acc 0.579
Epoch 03 | Train Loss 1.4748 | Train Acc 0.664 | Val Loss 1.3702 | Val Acc 0.684
Epoch 04 | Train Loss 1.1750 | Train Acc 0.765 | Val Loss 1.1115 | Val Acc 0.711
Epoch 05 | Train Loss 1.0516 | Train Acc 0.819 | Val Loss 0.9552 | Val Acc 0.789
Epoch 06 | Train Loss 0.8821 | Train Acc 0.839 | Val Loss 0.9503 | Val Acc 0.763
Epoch 07 | Train Loss 0.7892 | Train Acc 0.852 | Val Loss 0.7421 | Val Acc 0.842
Epoch 08 | Train Loss 0.6784 | Train Acc 0.859 | Val Loss 0.7427 | Val Acc 0.842
Epoch 09 | Train Loss 0.6188 | Train Acc 0.879 | Val Loss 0.7332 | Val Acc 0.842
Epoch 10 | Train Loss 0.5612 | Train Acc 0.899 | Val Loss 0.7358 | Val Acc 0.816
Epoch 11 | Train Loss 0.4975 | Train Acc 0.893 | Val Loss 0.6517 | Val Acc 0.842
Epoch 12 | Train Loss 0.4563 | Train Acc 0.906

In [15]:
run_live_webcam_inference()

I0000 00:00:1763302192.408854  192300 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1763302192.411793  192612 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.2.8-1), renderer: Mesa Intel(R) Xe Graphics (TGL GT2)
W0000 00:00:1763302192.434844  192603 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1763302192.466910  192605 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [77]:
demo_prediction_clip("/home/kaleab/Desktop/tiny_WLASL_dataset_processed/ACCEPT/65007.mp4")

I0000 00:00:1763198842.388845  200349 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1763198842.390232  224999 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.2.8-1), renderer: Mesa Intel(R) Xe Graphics (TGL GT2)
W0000 00:00:1763198842.396297  224989 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1763198842.405643  224995 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/kaleab/Documents/computer-vision/project/sign-language-detection/venv-computer-vision-project/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype(

Predicted word: ACCEPT (confidence 0.98)


## Usage Guide

1. **Set Paths**: Update `WordLevelConfig.videos_root` if your video dataset lives elsewhere.
2. **Precompute Landmarks**: Run `generate_sequence_cache(overwrite=True)` once to convert every video into landmark sequences.
3. **Train**: Execute `model = full_training_run()` to train and save the temporal CNN.
4. **Infer**: Use `demo_prediction_clip("/path/to/sample.mp4")` (or wire the helper into a live webcam script) to classify new clips.

Re-run the preprocessing step whenever you add new videos or change sampling parameters (e.g., `sequence_length`).

5. **Live Webcam**: After training, run `run_live_webcam_inference()` locally to stream predictions (press `q` to exit).
